In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/raw")

SESS_PKL = DATA_DIR / "ga_sessions.pkl"
SESS_CSV = DATA_DIR / "ga_sessions.csv"

HITS_PKL = DATA_DIR / "ga_hits-003.pkl"
HITS_CSV = DATA_DIR / "ga_hits-001.csv"


In [5]:
from pathlib import Path

DATA_DIR = Path("../data/raw")
sorted([p.name for p in DATA_DIR.iterdir()])


['ga_hits-001.csv', 'ga_hits-003.pkl', 'ga_sessions.csv', 'ga_sessions.pkl']

In [6]:
def read_best(path_pkl: Path, path_csv: Path) -> pd.DataFrame:
    if path_pkl.exists():
        return pd.read_pickle(path_pkl)
    return pd.read_csv(path_csv)

sessions = read_best(SESS_PKL, SESS_CSV)
hits = read_best(HITS_PKL, HITS_CSV)

print("sessions shape:", sessions.shape)
print("hits shape:", hits.shape)


sessions shape: (1860042, 18)
hits shape: (15726470, 11)


In [7]:
display(sessions.head(3))
display(hits.head(3))


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,device_category,device_os,device_brand,device_model,device_screen_resolution,device_browser,geo_country,geo_city
0,9055434745589932991.1637753792.1637753792,2108382700.1637753791,2021-11-24,14:36:32,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Zlatoust
1,905544597018549464.1636867290.1636867290,210838531.1636867288,2021-11-14,08:21:30,1,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,IGUCNvHlhfHpROGclCit,mobile,Android,Samsung,NaN,385x854,Samsung Internet,Russia,Moscow
2,9055446045651783499.1640648526.1640648526,2108385331.1640648523,2021-12-28,02:42:06,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,mobile,Android,Huawei,NaN,360x720,Chrome,Russia,Krasnoyarsk


,session_id,hit_date,hit_time,hit_number,hit_type,hit_referer,hit_page_path,event_category,event_action,event_label,event_value
0,5639623078712724064.1640254056.1640254056,2021-12-23,597864.0,30,event,NaN,sberauto.com/cars?utm_source_initial=google&ut...,quiz,quiz_show,NaN,None
1,7750352294969115059.1640271109.1640271109,2021-12-23,597331.0,41,event,NaN,sberauto.com/cars/fiat?city=1&city=18&rental_c...,quiz,quiz_show,NaN,None
2,885342191847998240.1640235807.1640235807,2021-12-23,796252.0,49,event,NaN,sberauto.com/cars/all/volkswagen/polo/e994838f...,quiz,quiz_show,NaN,None


In [13]:
# Функция: выбрать колонки по префиксу (например, utm_, device_)
def group_cols(cols, prefix):
    # Возвращаем отсортированный список названий колонок, которые начинаются с prefix
    return sorted([c for c in cols if c.startswith(prefix)])

# Собираем группы колонок по смысловым префиксам
groups = {
    "utm_": group_cols(sessions.columns, "utm_"),        # маркетинговые атрибуты
    "device_": group_cols(sessions.columns, "device_"),  # устройство пользователя
    "geo_": group_cols(sessions.columns, "geo_"),        # география пользователя
    "hit_": group_cols(hits.columns, "hit_"),            # данные о хите (просмотр/событие)
    "event_": group_cols(hits.columns, "event_"),        # данные о событии
}

# Быстро смотрим, сколько колонок в каждой группе
{g: len(v) for g, v in groups.items()}


{'utm_': 5, 'device_': 6, 'geo_': 2, 'hit_': 6, 'event_': 4}

In [9]:
groups


{'utm_': ['utm_adcontent',
  'utm_campaign',
  'utm_keyword',
  'utm_medium',
  'utm_source'],
 'device_': ['device_brand',
  'device_browser',
  'device_category',
  'device_model',
  'device_os',
  'device_screen_resolution'],
 'geo_': ['geo_city', 'geo_country'],
 'hit_': ['hit_date',
  'hit_number',
  'hit_page_path',
  'hit_referer',
  'hit_time',
  'hit_type'],
 'event_': ['event_action', 'event_category', 'event_label', 'event_value']}

In [14]:
# Функция отчёта по пропускам: сколько и какая доля NaN по каждой колонке
def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)  # число строк в датасете
    
    # Считаем количество пропусков и долю пропусков по каждой колонке
    rep = pd.DataFrame({
        "column": df.columns,                           # название колонки
        "missing_cnt": df.isna().sum().values,          # количество NaN
        "missing_rate": (df.isna().sum().values / n).round(4),  # доля NaN
        "dtype": [str(t) for t in df.dtypes],           # тип pandas
    }).sort_values("missing_rate", ascending=False)     # сортируем по доле пропусков
    
    return rep.reset_index(drop=True)  # сбрасываем индекс для красоты

# Отчёт по пропускам для sessions
miss_sessions = missing_report(sessions)

# Отчёт по пропускам для hits
miss_hits = missing_report(hits)

# Показываем топ-15 колонок с максимальной долей пропусков
display(miss_sessions.head(15))
display(miss_hits.head(15))


,column,missing_cnt,missing_rate,dtype
0,device_model,1843704,0.9912,object
1,utm_keyword,1082061,0.5817,object
2,device_os,1070138,0.5753,object
3,utm_adcontent,335615,0.1804,object
4,utm_campaign,219603,0.1181,object
5,device_brand,118678,0.0638,object
6,utm_source,97,0.0001,object
7,geo_country,0,0.0000,object
8,device_browser,0,0.0000,object
9,device_screen_resolution,0,0.0000,object


,column,missing_cnt,missing_rate,dtype
0,event_value,15726470,1.0000,object
1,hit_time,9160322,0.5825,float64
2,hit_referer,6274804,0.3990,object
3,event_label,3760184,0.2391,object
4,session_id,0,0.0000,object
5,hit_date,0,0.0000,object
6,hit_number,0,0.0000,int64
7,hit_type,0,0.0000,object
8,hit_page_path,0,0.0000,object
9,event_category,0,0.0000,object


In [15]:
# Проверяем, сколько дубликатов session_id в sessions
# (в идеале session_id уникален в таблице sessions)
print("sessions duplicated session_id:", sessions["session_id"].duplicated().sum())



sessions duplicated session_id: 0


In [16]:
# Для hits часто ключ = (session_id, hit_number)
# Проверяем, сколько дубликатов по этой паре
print(
    "hits duplicated by ['session_id','hit_number']:",
    hits.duplicated(subset=["session_id", "hit_number"]).sum()
)


hits duplicated by ['session_id','hit_number']: 225862


In [17]:
# Приводим visit_date к datetime (не меняя исходный df), чтобы проверить min/max
visit_date_dt = pd.to_datetime(sessions["visit_date"], errors="coerce")  # ошибки → NaT

# Приводим hit_date к datetime (не меняя исходный df), чтобы проверить min/max
hit_date_dt = pd.to_datetime(hits["hit_date"], errors="coerce")  # ошибки → NaT

# Печатаем диапазоны (минимум/максимум)
print("sessions visit_date min/max:", visit_date_dt.min(), visit_date_dt.max())
print("hits hit_date min/max:", hit_date_dt.min(), hit_date_dt.max())

# Дополнительно можно посмотреть, сколько некорректных дат превратилось в NaT
print("sessions visit_date NaT:", visit_date_dt.isna().sum())
print("hits hit_date NaT:", hit_date_dt.isna().sum())


sessions visit_date min/max: 2021-05-19 00:00:00 2021-12-31 00:00:00
hits hit_date min/max: 2021-05-19 00:00:00 2021-12-31 00:00:00
sessions visit_date NaT: 0
hits hit_date NaT: 0


In [18]:
# Функция: доля пустых значений (NaN или пустая/пробельная строка)
def empty_rate(series: pd.Series) -> float:
    # Приводим к строковому типу pandas
    s = series.astype("string")
    # Пустое = NaN или строка после trim равна ""
    return float(((s.isna()) | (s.str.strip() == "")).mean())

# Список колонок utm_* в sessions
utm_cols = [c for c in sessions.columns if c.startswith("utm_")]

# Список колонок event_* в hits
event_cols = [c for c in hits.columns if c.startswith("event_")]

# Считаем доли пустых по utm_*
utm_empty = (
    pd.DataFrame({
        "column": utm_cols,
        "empty_rate": [empty_rate(sessions[c]) for c in utm_cols]
    })
    .sort_values("empty_rate", ascending=False)
)

# Считаем доли пустых по event_*
event_empty = (
    pd.DataFrame({
        "column": event_cols,
        "empty_rate": [empty_rate(hits[c]) for c in event_cols]
    })
    .sort_values("empty_rate", ascending=False)
)

# Показываем результаты
display(utm_empty)
display(event_empty)


,column,empty_rate
4,utm_keyword,0.581740
3,utm_adcontent,0.180434
2,utm_campaign,0.118063
0,utm_source,0.000052
1,utm_medium,0.000000


,column,empty_rate
3,event_value,1.000000
2,event_label,0.239099
0,event_category,0.000000
1,event_action,0.000000


In [19]:
# Делаем копии, чтобы не портить исходные данные
sessions_clean = sessions.copy()
hits_clean = hits.copy()

# --- Даты ---
# Приводим visit_date к datetime
sessions_clean["visit_date"] = pd.to_datetime(sessions_clean["visit_date"], errors="coerce")

# Приводим hit_date к datetime
hits_clean["hit_date"] = pd.to_datetime(hits_clean["hit_date"], errors="coerce")

# --- Время ---
# Оставляем время строкой (так проще для дальнейшей загрузки в SQLite и анализа)
sessions_clean["visit_time"] = sessions_clean["visit_time"].astype("string").str.strip()
hits_clean["hit_time"] = hits_clean["hit_time"].astype("string").str.strip()

# --- Числа / порядковые номера ---
# visit_number приводим к целому nullable Int64
sessions_clean["visit_number"] = pd.to_numeric(sessions_clean["visit_number"], errors="coerce").astype("Int64")

# hit_number приводим к целому nullable Int64
hits_clean["hit_number"] = pd.to_numeric(hits_clean["hit_number"], errors="coerce").astype("Int64")

# event_value если есть — к числу (float/nullable)
if "event_value" in hits_clean.columns:
    hits_clean["event_value"] = pd.to_numeric(hits_clean["event_value"], errors="coerce")

# --- Trim для всех строковых колонок ---
# В sessions: все object-колонки приводим к string и чистим пробелы
for c in sessions_clean.select_dtypes(include=["object"]).columns:
    sessions_clean[c] = sessions_clean[c].astype("string").str.strip()

# В hits: все object-колонки приводим к string и чистим пробелы
for c in hits_clean.select_dtypes(include=["object"]).columns:
    hits_clean[c] = hits_clean[c].astype("string").str.strip()

# Смотрим типы после приведения (проверка)
sessions_clean.dtypes


session_id                  string[python]
client_id                   string[python]
visit_date                  datetime64[ns]
visit_time                  string[python]
visit_number                         Int64
utm_source                  string[python]
utm_medium                  string[python]
utm_campaign                string[python]
utm_adcontent               string[python]
utm_keyword                 string[python]
device_category             string[python]
device_os                   string[python]
device_brand                string[python]
device_model                string[python]
device_screen_resolution    string[python]
device_browser              string[python]
geo_country                 string[python]
geo_city                    string[python]
dtype: object

In [20]:
# Типы hits после приведения (проверка)
hits_clean.dtypes

session_id        string[python]
hit_date          datetime64[ns]
hit_time          string[python]
hit_number                 Int64
hit_type          string[python]
hit_referer       string[python]
hit_page_path     string[python]
event_category    string[python]
event_action      string[python]
event_label       string[python]
event_value              float64
dtype: object

In [24]:
# Смотрим, являются ли эти строки полными дублями (по всем колонкам)
full_dup_cnt = hits.duplicated().sum()
print("hits full duplicated rows (all columns):", full_dup_cnt)

# Смотрим пример одной проблемной пары (session_id, hit_number) где есть повторы
dup_pairs = hits[hits.duplicated(subset=["session_id","hit_number"], keep=False)] \
    .sort_values(["session_id","hit_number"])

display(dup_pairs.head(20))


hits full duplicated rows (all columns): 0


,session_id,hit_date,hit_time,hit_number,hit_type,hit_referer,hit_page_path,event_category,event_action,event_label,event_value
13161439,1000457743553563495.1632406504.1632406504,2021-09-23,1018.0,3,event,NaN,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,<NA>,<NA>
13612179,1000457743553563495.1632406504.1632406504,2021-09-23,NaN,3,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,KclpemfoHstknWHFiLit,<NA>
10458798,1000457743553563495.1632406504.1632406504,2021-09-23,NaN,4,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_new_card,KclpemfoHstknWHFiLit,<NA>
11809677,1000457743553563495.1632406504.1632406504,2021-09-23,1040.0,4,event,NaN,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_new_card,<NA>,<NA>
11359619,1000457743553563495.1632406504.1632406504,2021-09-23,NaN,5,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,KclpemfoHstknWHFiLit,<NA>
12710380,1000457743553563495.1632406504.1632406504,2021-09-23,1043.0,5,event,NaN,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,<NA>,<NA>
13613387,1000457743553563495.1632406504.1632406504,2021-09-23,NaN,6,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_new_card,KclpemfoHstknWHFiLit,<NA>
14063541,1000457743553563495.1632406504.1632406504,2021-09-23,1045.0,6,event,NaN,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_new_card,<NA>,<NA>
10458026,1000457743553563495.1632406504.1632406504,2021-09-23,NaN,7,event,HbolMJUevblAbkHClEQa,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,KclpemfoHstknWHFiLit,<NA>
11359013,1000457743553563495.1632406504.1632406504,2021-09-23,1092.0,7,event,NaN,sberauto.com/cars/6afb1543?rental_page=rental_car,card_web,view_card,<NA>,<NA>


In [25]:
# Сколько строк на одну пару (session_id, hit_number)
pair_counts = hits.groupby(["session_id","hit_number"]).size().sort_values(ascending=False)

print("max rows per (session_id, hit_number):", pair_counts.iloc[0])
display(pair_counts.head(10))


max rows per (session_id, hit_number): 2


session_id                                 hit_number
4559031254314091707.1632339133.1632339133  50            2
3711148884098832220.1632393052.1632393052  16            2
                                           3             2
                                           4             2
                                           7             2
                                           10            2
                                           11            2
                                           12            2
                                           13            2
                                           14            2
dtype: int64

In [26]:
# Копии для "чистых" версий (чтобы дальше использовать их для EDA/SQLite)
sessions_clean = sessions.copy()
hits_clean = hits.copy()

# --- Dates ---
sessions_clean["visit_date"] = pd.to_datetime(sessions_clean["visit_date"], errors="coerce")
hits_clean["hit_date"] = pd.to_datetime(hits_clean["hit_date"], errors="coerce")

# --- Times ---
# visit_time оставляем строкой HH:MM:SS (SQLite-friendly)
sessions_clean["visit_time"] = sessions_clean["visit_time"].astype("string").str.strip()

# hit_time похоже на числовой показатель (секунды/смещение), поэтому приводим к целому nullable
hits_clean["hit_time"] = pd.to_numeric(hits_clean["hit_time"], errors="coerce").astype("Int64")

# --- Numbers ---
sessions_clean["visit_number"] = pd.to_numeric(sessions_clean["visit_number"], errors="coerce").astype("Int64")
hits_clean["hit_number"] = pd.to_numeric(hits_clean["hit_number"], errors="coerce").astype("Int64")

# --- Drop useless колонок ---
# event_value = 100% пусто → удаляем
if "event_value" in hits_clean.columns:
    hits_clean = hits_clean.drop(columns=["event_value"])

# --- Trim all string columns ---
for c in sessions_clean.select_dtypes(include=["object"]).columns:
    sessions_clean[c] = sessions_clean[c].astype("string").str.strip()

for c in hits_clean.select_dtypes(include=["object"]).columns:
    hits_clean[c] = hits_clean[c].astype("string").str.strip()

# Проверка типов
display(sessions_clean.dtypes)
display(hits_clean.dtypes)


session_id                  string[python]
client_id                   string[python]
visit_date                  datetime64[ns]
visit_time                  string[python]
visit_number                         Int64
utm_source                  string[python]
utm_medium                  string[python]
utm_campaign                string[python]
utm_adcontent               string[python]
utm_keyword                 string[python]
device_category             string[python]
device_os                   string[python]
device_brand                string[python]
device_model                string[python]
device_screen_resolution    string[python]
device_browser              string[python]
geo_country                 string[python]
geo_city                    string[python]
dtype: object

session_id        string[python]
hit_date          datetime64[ns]
hit_time                   Int64
hit_number                 Int64
hit_type          string[python]
hit_referer       string[python]
hit_page_path     string[python]
event_category    string[python]
event_action      string[python]
event_label       string[python]
dtype: object

In [27]:
display(missing_report(sessions_clean).head(10))
display(missing_report(hits_clean).head(10))


,column,missing_cnt,missing_rate,dtype
0,device_model,1843704,0.9912,string
1,utm_keyword,1082061,0.5817,string
2,device_os,1070138,0.5753,string
3,utm_adcontent,335615,0.1804,string
4,utm_campaign,219603,0.1181,string
5,device_brand,118678,0.0638,string
6,utm_source,97,0.0001,string
7,geo_country,0,0.0000,string
8,device_browser,0,0.0000,string
9,device_screen_resolution,0,0.0000,string


,column,missing_cnt,missing_rate,dtype
0,hit_time,9160322,0.5825,Int64
1,hit_referer,6274804,0.3990,string
2,event_label,3760184,0.2391,string
3,session_id,0,0.0000,string
4,hit_date,0,0.0000,datetime64[ns]
5,hit_number,0,0.0000,Int64
6,hit_type,0,0.0000,string
7,hit_page_path,0,0.0000,string
8,event_category,0,0.0000,string
9,event_action,0,0.0000,string


# Этап 1. Подготовительная работа — выводы (≈ 1 час)

## 1) Загрузка данных
**Файлы:**
- `ga_sessions.pkl / ga_sessions.csv`
- `ga_hits-003.pkl / ga_hits-001.csv`

**Почему два формата (PKL и CSV):**
- `PKL` читается быстрее и сохраняет типы pandas (удобно для ноутбука).
- `CSV` — универсальный формат для обмена/БД/проверок, но медленнее и часто теряет типы.

---

## 2) Схема и группы атрибутов
**ga_sessions (сессии):**
- идентификаторы: `session_id`, `client_id`
- время визита: `visit_date`, `visit_time`, `visit_number`
- маркетинг: `utm_*` (source/medium/campaign/adcontent/keyword)
- устройство: `device_*` (category/os/brand/model/screen/browser)
- география: `geo_*` (country/city)

**ga_hits (хиты/события):**
- идентификатор сессии: `session_id`
- время хита: `hit_date`, `hit_time`, `hit_number`
- hit-метаданные: `hit_type`, `hit_referer`, `hit_page_path`
- события: `event_*` (category/action/label/value)

**Проверка группировок по префиксам:**
- `utm_`: 5 колонок  
- `device_`: 6 колонок  
- `geo_`: 2 колонки  
- `hit_`: 6 колонок  
- `event_`: 4 колонки  

---

## 3) Качество данных: пропуски и дубликаты
### Пропуски (top проблемных полей)
**ga_sessions:**
- `device_model` ≈ 99% пропусков → поле почти бесполезно для анализа/ML
- `utm_keyword` ≈ 58% пропусков → допустимо (часто нет keyword)
- `device_os` ≈ 57% пропусков → оставляем NULL, проверим на EDA (возможные особенности трекинга)
- `utm_adcontent` ≈ 18%, `utm_campaign` ≈ 12%, `device_brand` ≈ 6% → оставляем NULL

**ga_hits:**
- `event_value` = 100% пропусков → поле исключаем из дальнейшей работы/БД
- `hit_time` ≈ 58% пропусков → приводим к числовому nullable-типу, пропуски оставляем NULL
- `hit_referer` ≈ 40% пропусков → норм (часто отсутствует)
- `event_label` ≈ 24% пропусков → оставляем NULL

### Дубликаты
- `ga_sessions.session_id`: **0** дублей (ключ уникален)
- `ga_hits`:
  - полных дублей строк: **0**
  - повторов по паре (`session_id`, `hit_number`): **225 862**
  - максимум строк на одну пару (`session_id`, `hit_number`): **2**
  - вывод: это **не ошибка**, а реальные разные хиты с одинаковым `hit_number` (например, различаются `hit_time`, `hit_referer`, `event_label`)

---

## 4) Sanity checks
- диапазон дат:
  - `visit_date`: 2021-05-19 … 2021-12-31
  - `hit_date`: 2021-05-19 … 2021-12-31
- некорректных дат (NaT): **0**

---

## 5) Принятые правила подготовки данных (для EDA и загрузки в SQLite)
**Общие:**
- пропуски оставляем как `NULL` (NaN), без заполнения `"unknown"` на этапе подготовки
- строки: trim пробелов и приведение к pandas `string`
- даты: `visit_date`, `hit_date` → `datetime`

**ga_sessions:**
- `visit_number` → nullable `Int64`
- `visit_time` оставляем строкой `HH:MM:SS`
- `device_model` можно не переносить в SQLite (≈99% NULL) — решение для оптимизации объёма

**ga_hits:**
- `hit_number` → nullable `Int64`
- `hit_time` → nullable `Int64` (как числовой показатель), пропуски → NULL
- `event_value` удаляем (100% NULL)
- ключ в БД: не используем (`session_id`, `hit_number`) как PK → добавляем суррогатный `hit_id` (AUTOINCREMENT) при загрузке в SQLite

---

## Артефакт этапа
- `notebooks/01_data_loading_cleaning.ipynb`: загрузка → группировка атрибутов → отчёты по пропускам/дубликатам → правила типизации/очистки
- Подготовлены `sessions_clean`, `hits_clean` для этапа EDA и последующей загрузки в SQLite

- ---

## Риски / что проверить на Этапе 2 (EDA)
Чтобы не сделать неверных выводов и корректно подготовить данные к БД/ML, на EDA нужно отдельно проверить:

### 1) `device_os` (≈57% пропусков) — почему так много?
- Сравнить долю пропусков `device_os` по `device_category` (mobile/desktop/tablet).
- Проверить связку `device_browser` ↔ `device_os`: встречаются ли браузеры без OS системно.
- Посмотреть топ комбинаций `device_category + device_brand + device_browser` среди строк, где `device_os` пустой.
**Цель:** понять, это “нормальная особенность трекинга” или данные сломаны/частично потеряны.

### 2) `hit_time` (≈58% пропусков, числовой формат) — что это за время?
- Проверить распределение `hit_time` (min/max, квантили) и наличие “аномально больших” значений.
- Сравнить `hit_time` по `hit_type` (например, у `event` vs других типов).
- Для пары строк с одинаковым (`session_id`, `hit_number`) сравнить `hit_time`: часто ли отличается и как.
**Цель:** интерпретировать `hit_time` (смещение, секунды, порядковый таймстемп) и решить, как хранить/использовать.

### 3) Повторы по (`session_id`, `hit_number`) — влияет ли это на метрики?
- Проверить, какие поля чаще всего отличаются в повторяющихся парах (например, `hit_referer`, `event_label`, `hit_time`).
- Оценить, как это влияет на агрегаты “хитов на сессию” и “событий на сессию”.
**Цель:** корректно считать метрики без ложного “удвоения” или потери событий.

### 4) Маркетинговые поля `utm_*` — где пропуски “нормальные”, а где подозрительные?
- Доля пустых `utm_*` по источникам трафика (например, `utm_source` заполнен, а `utm_campaign` пустой).
- Группировка по `utm_medium` (banner/cpm/organic и т.п.) и сравнение заполненности.
**Цель:** правильно интерпретировать “органику/прямой трафик” vs “платные кампании”.

### 5) География и “шум” в строках
- Проверить нормальность значений `geo_country`, топ городов, наличие странных/пустых строк (после trim).
**Цель:** подготовить качественные категориальные признаки и избежать “мусорных” категорий.

---

